In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

clawpack_available = True
try:
    from clawpack.visclaw import geoplot
except ImportError:
    clawpack_available = False

fft = np.fft.fft
ifft =np.fft.ifft
fftfreq = np.fft.fftfreq

cos = np.cos
sin = np.sin
tanh = np.tanh

sqrt = np.sqrt

In [ ]:
import matplotlib as mpl
from IPython.display import HTML
mpl.rcParams['animation.embed_limit'] = 200  # in MB
def plotting_func(frames,times,x,h,xlims=None,plotbath=True,modes=None):
    # print("Checking shapes:")
    # print(f"frames: {frames.shape}, times: {times.shape}, x: {x.shape}, h: {h}")
    print("Creating animation...")
    fig, ax = plt.subplots(figsize=(10, 4))
    line_eta, = ax.plot([], [], color="#117af1", label='eta')
    line_bath, = ax.plot([], [], color='black', label='bath')
    title = ax.set_title("")
    maxeta = np.max(np.abs(frames[:,0,:]))
    L = x[-1] - x[0]

    color_list = ["#f111f1", "#f11111", "#030405", "#00FC4C"]
    if modes is not None:
        def phase_velocity(xi):
            return np.sqrt(g * np.tanh(xi * h) / xi)

        phase_velocities = [phase_velocity(xi[mode]) for mode in modes]
        mode_plots = []
        for i, mode in enumerate(modes):
            sc = ax.scatter([x[0]], [frames[0,0,0]],
                            color=color_list[i % len(color_list)],
                            label=f"Mode {mode}",zorder=10,
                            s=100, edgecolor='black')
            mode_plots.append(sc)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("Surface Elevation (m)")
    # Before update()
    if plotbath:
        fill_bath = ax.fill_between([], [], [], color='lightgrey', zorder=0)
    fill_eta = ax.fill_between([], [], [], color='#b3d9ff', zorder=1)
    if xlims is not None:
        ax.set_xlim(xlims)
    else:
        ax.set_xlim(np.min(x), np.max(x))

    def update(frame):
        nonlocal fill_bath, fill_eta, mode_plots
        eta = frames[frame,0]
        bath = -h*np.ones_like(eta)

        line_eta.set_data(x, eta)
        line_bath.set_data(x, bath)

        # Remove previous fills
        if plotbath:
            fill_bath.remove()

        fill_eta.remove()

        # Draw new fills and save references
        if plotbath:
            fill_bath = ax.fill_between(x, bath, bath - 0.1*h, color='lightgrey', zorder=0)
        if modes is not None:
            for i, sc in enumerate(mode_plots):
                xmode = phase_velocities[i] * times[frame] % (L)-L/2.
                closest_index = np.argmin(np.abs(x - xmode))
                sc.set_offsets([[x[closest_index], frames[frame, 0, closest_index]]])
        fill_eta = ax.fill_between(x, eta, bath, where=~np.isnan(eta) & ~np.isnan(bath),
                                color='#b3d9ff', zorder=1)

        title.set_text(f"g= {g:.2f} $m/s^2$, h = {h:.0f} m, t = {times[frame]:.2f} s")

        if plotbath:
            ax.set_ylim(min(bath) - 0.1*h, maxeta + 0.1*maxeta)
            return line_eta, line_bath, title
        else:
            ax.set_ylim(-maxeta - 0.1*maxeta, maxeta + 0.1*maxeta)
            return line_eta, title

    # === Create animation ===
    ani = animation.FuncAnimation(
        fig, update, frames=len(frames), blit=False, interval=100
    )

    # === Save to mp4 ===
    # writer = animation.FFMpegWriter(fps=20, metadata=dict(artist='Your Name'))
    # ani.save("animation.mp4", writer=writer)
    #return HTML plot
    return HTML(ani.to_jshtml())

def animate_2d(frames, times, X, Y, h, clim=None, xlims=None, ylims=None):
    """
    Animate 2D solution frames with imshow (fast + stable).
    
    frames: (n_frames, 2, mx, my)   solution data
    times:  (n_frames,)             time array
    X, Y:   meshgrid arrays          spatial grids
    h:      float                   depth (for labeling)
    clim:   (vmin, vmax) or None    color limits
    """

    print("Creating 2D animation...")

    fig, ax = plt.subplots(figsize=(6,6))

    # Set color limits
    if clim is None:
        vmax = np.max(np.abs(frames[10:,0]))
        clim = (-vmax, vmax)

    # Initial imshow
    extent = [X.min(), X.max(), Y.min(), Y.max()]
    if clawpack_available:
        im = ax.imshow(frames[0,0], extent=extent, cmap=geoplot.tsunami_colormap,
                   origin="lower", vmin=clim[0], vmax=clim[1])
    else:
        im = ax.imshow(frames[0,0], extent=extent, cmap="viridis",
                   origin="lower", vmin=clim[0], vmax=clim[1])
    # Colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Surface Elevation (m)", fontsize=12)

    # Axes limits
    if xlims is not None:
        ax.set_xlim(xlims)
    if ylims is not None:
        ax.set_ylim(ylims)
    ax.set_aspect("equal")

    # Title
    title = ax.set_title(f"t = {times[0]:.2f} s, h = {h} m")

    # Update function
    def update(i):
        im.set_data(frames[i,0])   # update image
        title.set_text(f"t = {times[i]:.2f} s, h = {h} m")
        return [im, title]

    ani = animation.FuncAnimation(fig, update,
                                  frames=len(frames),
                                  blit=False, interval=100)

    return HTML(ani.to_jshtml())

### The 1D linear water waves problem
The linear water wave equations with a flat bottom are given by
$$
\begin{align*}
\begin{cases}
\partial_t \eta &=  D\psi, \\
\partial_t \psi &= -g\eta.
\end{cases}
\end{align*}
$$
Here, $\eta(t,x,y)$ is the surface elevation, $\psi(t,x,y)$ is the velocity potential evaluated at the surface, 
and $D$ is the Dirichlet to Neumann (DtN) operator  with Fourier symbol
$$
s_D(\xi) = |\xi| \tanh(h|\xi|).
$$

We use a simple spectral method implementation to solve these equations:
1. Given the collocated initial conditions $\eta_0$ and $\psi_0$, compute the Fourier transforms $\hat{\eta}_0$ and $\hat{\psi}_0$.
2. For each time step, compute the exponential of the local linear operator $t L$, where
    $$
    L = \begin{pmatrix}
    0 & s_D(\xi) \\
    -g & 0
    \end{pmatrix}
    $$
    The latter can be computed explicitly as
    $$
    \exp(t L) = \begin{pmatrix}
    \cos(t\omega(\xi)) & s_D(\xi)\frac{\sin(t\omega(\xi))}{\omega(\xi)} \\
    -g\frac{\sin(t\omega(\xi))}{\omega(\xi)} & \cos(t\omega(\xi))
    \end{pmatrix},
    $$
    where $\omega(\xi) = \sqrt{g s_D(\xi)}$ is the linear dispersion relation of the water waves problem.
3. Apply $\exp(t L)$ to $[\eta_0, \psi_0]^T$ and take the inverse Fourier transform to obtain $\eta(t)$ and $\psi(t)$.




In [ ]:
h=1.
g=9.81
m = 2**12 
L = 1000.
n_frames = 100
tfinal = 50.

x = np.linspace(-L/2., L/2., m)
times = np.linspace(0, tfinal, n_frames)
xi = 2*np.pi*np.fft.fftfreq(m, d=L/m)

eta = np.exp(-x**2)
psi = np.zeros_like(x)

IC_hat = np.vstack([eta,psi])
IC_hat = fft(IC_hat,axis=1)

xi=abs(xi)

sol_hat = np.zeros((2, m))
sD = xi* np.tanh(xi * h)
exptL = np.zeros((2,2,x.size),dtype=np.complex128)

frames = []

omega = np.sqrt(g * sD) #Linear dispersion relation

for t in times:
    safe_sinc = np.where(omega>1e-14, sin(t*omega)/omega, t)
    exptL[0, 0, :] = cos(t*omega)
    exptL[0, 1, :] = sD*safe_sinc
    exptL[1, 0, :] = -g*safe_sinc
    exptL[1, 1, :] = cos(t*omega)
    frames.append(ifft(np.einsum('ijk,jk->ik', exptL, IC_hat),axis=1).real)

frames = np.array(frames)


In [ ]:
plot = plotting_func(frames, times, x, h, xlims=(-100.,100.))
plt.close()
plot

In [ ]:
h=1000.
g=9.81
m = 2**12 
L = 5000.
n_frames = 100
tfinal = 120.

x = np.linspace(-L/2., L/2., m)
times = np.linspace(0, tfinal, n_frames)
xi =np.abs(np.fft.fftfreq(m)*m*2*np.pi/L)  # Wavenumber "grid"

eta_hat = np.zeros_like(xi)

modes = [1,2,3,4]
# modes = [1,50]

for i,mode in enumerate(modes):
    eta_hat[mode] = m*(1-(i)/len(modes))/len(modes) # Set the first four Fourier modes to 1
    eta_hat[mode]/(i+1)

# modes.append(2**11)
# eta_hat[2**11] = 0.01*m/len(modes) 

eta = ifft(eta_hat).real
psi = np.zeros_like(x)

IC_hat = np.vstack([eta,psi])
IC_hat = fft(IC_hat,axis=1)

sol_hat = np.zeros((2, m))
sD = xi* np.tanh(xi * h)
exptL = np.zeros((2,2,x.size),dtype=np.complex128)

frames = []

omega = np.sqrt(g * sD) #Linear dispersion relation

for t in times:
    safe_sinc = np.where(omega>1e-14, sin(t*omega)/omega, t)
    exptL[0, 0, :] = cos(t*omega)
    exptL[0, 1, :] = sD*safe_sinc
    exptL[1, 0, :] = -g*safe_sinc
    exptL[1, 1, :] = cos(t*omega)
    frames.append(ifft(np.einsum('ijk,jk->ik', exptL, IC_hat),axis=1).real)

frames = np.array(frames)

In [ ]:
plot = plotting_func(frames, times, x, h,plotbath=False,modes=modes)
plt.close()  
plot

### The 2D linear water waves problem
The DtN operator is the same as in 1D, as it depends only on $|\xi|$.

In [ ]:
h=100.
g=9.81
m = 2**10
n_frames = 20
tfinal = 200.

# Domain
Lx, Ly = 20000., 20000.        # box size in x and y
mx, my = m, m        # number of grid points in x and y

x = np.linspace(-Lx/2, Lx/2, mx, endpoint=False)
y = np.linspace(-Ly/2, Ly/2, my, endpoint=False)
X, Y = np.meshgrid(x, y, indexing="ij")

# Time discretization
times = np.linspace(0, tfinal, n_frames)

# Fourier wavenumber grids
xi_x = 2*np.pi*np.fft.fftfreq(mx, d=Lx/mx)   # spacing in x
xi_y = 2*np.pi*np.fft.fftfreq(my, d=Ly/my)   # spacing in y
XI_x, XI_y = np.meshgrid(xi_x, xi_y, indexing="ij")
XI = np.sqrt(XI_x**2 + XI_y**2)             # radial wavenumber if needed

# Initial conditions (2 fields in 2D)
width = 100.
r0 = 0.
ampl = 5.
r = np.sqrt((X - r0)**2 + (Y - r0)**2)
eta = ampl * np.exp(-r**2 / width**2)               # Gaussian in 2D
psi = np.zeros_like(X)

IC = np.stack([eta, psi])                  # shape (2, mx, my)

# Fourier transform
IC_hat = np.fft.fft2(IC, axes=(1,2))       # FFT along spatial dims

# Dispersion
sD = XI * np.tanh(XI * h)
omega = np.sqrt(g * sD) #Linear dispersion relation

frames = []
for t in times:
    safe_sinc = np.where(omega>1e-14, sin(t*omega)/omega, t)
    exptL = np.zeros((2,2,mx,my), dtype=complex)
    exptL[0, 0, :, :] = cos(t*omega)
    exptL[0, 1, :, :] = sD*safe_sinc
    exptL[1, 0, :, :] = -g*safe_sinc
    exptL[1, 1, :, :] = cos(t*omega)

    sol_hat = np.einsum("ijxy,jxy->ixy", exptL, IC_hat)
    sol = np.fft.ifft2(sol_hat, axes=(1,2)).real

    frames.append(sol)

frames = np.array(frames)   # shape (n_frames, 2, mx, my)



In [ ]:
plot2d = animate_2d(frames, times, X, Y, h, xlims=(0., 5000.), ylims=(0., 5000.),clim=(-0.1,0.1))

In [ ]:
plot2d

In [ ]:
#Dump into npz file
np.savez("linear_2d_wave_data.npz", frames=frames, times=times, X=X, Y=Y, h=h, g=g)